# 요구사항 데이터셋 연구 EDA

이 노트북은 단순히 문서별 행 수를 세는 대신, **LLM 라벨 판단과 후속 모델 평가에 영향을 줄 수 있는 데이터 특성**을 탐색합니다. 분석 대상은 동결 데이터셋 `requirements_v0.2.0` 1,024건과 라벨링 파일럿 표본 40건입니다.

## 핵심 질문

1. 비용·범위·책임·검수·완화 조건이 데이터에 어떻게 분포하는가?
2. 개별 요구사항 한 행만으로 판단하기 어려운 문맥 의존 후보는 얼마나 되는가?
3. 문구는 비슷하지만 판단 조건이 다른 요구사항이 존재하는가?
4. 40건 파일럿은 문서 비율이 아니라 **판단 요소와 난이도 공간**을 충분히 덮는가?

## 해석 원칙

- 아래 규칙 기반 신호는 정답 라벨이 아니라 사람 감사 대상을 찾기 위한 탐색 피처입니다.
- `EDA용 통합 유형`은 기관마다 다른 원문 요구사항 유형을 비교하기 위한 편의상 대분류이며 공식 표준이 아닙니다.
- TF-IDF 유사도는 의미적 동일성을 보장하지 않습니다. 제공 주체·수량·상한·비용 부담·검수 기준 차이를 원문에서 확인해야 합니다.
- 행 수는 학습 가중치와 평가 분산을 설명하는 보조 정보로만 사용하고, 핵심 결론으로 해석하지 않습니다.

In [ ]:
from pathlib import Path
import re
import sys

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from scripts.data.eda_requirements import normalize_requirement_type

DATASET_PATH = ROOT / 'data/processed/requirements_v0.2.0.jsonl'
PILOT_PATH = ROOT / 'data/samples/labeling_pilot_sample_v0.1.0.jsonl'
df = pd.read_json(DATASET_PATH, lines=True)
pilot = pd.read_json(PILOT_PATH, lines=True)

FACTOR_PATTERNS = {
    '비용요인': r'상주|투입.{0,6}인력|전담.{0,6}인력|장비|서버|GPU|라이선스|교육|유지보수|운영지원|클라우드|API.{0,4}비용',
    '범위수량': r'(?:최소|최대|이상|이하|이내|초과)|[0-9][0-9,]*(?:\.[0-9]+)?\s*(?:명|식|개|대|회|개월|년|TB|GB|건)',
    '범위모호': r'협의|추후.{0,8}확정|필요시|요청에 따라|별도.{0,8}정|일체|기타.{0,8}사항',
    '책임위험': r'무상|귀책|배상|지체상금|손해|하자|수행사.{0,8}책임|사업자.{0,8}책임',
    '검수기준': r'검수|합격.{0,8}기준|성능.{0,8}(?:기준|보장)|정확도|시험.{0,8}결과|평가.{0,8}지표|골든셋',
    '완화조건': r'(?:발주처|위원회|공사|기관).{0,12}제공|별도.{0,6}계약|범위.{0,6}한정|상한|유상|추가.{0,6}비용',
}
CONTEXT_PATTERNS = {
    '외부참조': r'상기|전술|아래와 같이|별첨|붙임|부록|제안요청서.{0,6}참조|[A-Z]{2,8}(?:-[A-Z]+)*-+[0-9]{3}',
    '결정유보': r'협의|추후.{0,8}확정|기술협상|착수.{0,6}후|사업.{0,6}중.{0,6}결정',
    '불특정범위': r'필요시|요청에 따라|관련.{0,8}사항|기타.{0,8}사항|등을 포함|일체',
}

def compact_text(value):
    return re.sub(r'\s+', ' ', str(value)).strip()

def prepare_frame(frame, length_edges=None):
    result = frame.copy()
    result['eda_type_group'] = result['requirement_type'].apply(normalize_requirement_type)
    result['analysis_text'] = result['raw_requirement_text'].map(compact_text)
    result['char_len'] = result['analysis_text'].str.len()
    result['word_len'] = result['analysis_text'].str.split().str.len()
    for name, pattern in {**FACTOR_PATTERNS, **CONTEXT_PATTERNS}.items():
        result[name] = result['analysis_text'].str.contains(pattern, regex=True, na=False)
    result['판단요소수'] = result[list(FACTOR_PATTERNS)].sum(axis=1)
    result['문맥신호수'] = result[list(CONTEXT_PATTERNS)].sum(axis=1)
    result['문맥검토후보'] = result['외부참조'] | (result['문맥신호수'] >= 2)
    if length_edges is not None:
        result['length_group'] = pd.cut(
            result['char_len'], bins=length_edges,
            labels=['짧음(≤P50)', '중간(P50–P90)', '장문(>P90)'], include_lowest=True,
        )
    return result

df = prepare_frame(df)
length_edges = [-np.inf, df['char_len'].quantile(.5), df['char_len'].quantile(.9), np.inf]
df['length_group'] = pd.cut(df['char_len'], bins=length_edges, labels=['짧음(≤P50)', '중간(P50–P90)', '장문(>P90)'], include_lowest=True)
pilot = prepare_frame(pilot, length_edges=length_edges)

sns.set_theme(style='whitegrid')
installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
font_candidates = ['Malgun Gothic', 'NanumGothic', 'Noto Sans CJK KR', 'AppleGothic', 'DejaVu Sans']
selected_font = next(font for font in font_candidates if font in installed_fonts)
plt.rcParams['font.family'] = selected_font
plt.rcParams['axes.unicode_minus'] = False
print(f'matplotlib font={selected_font}')
print(f'dataset={len(df):,} rows / {df.document_id.nunique()} documents')
print(f'pilot={len(pilot):,} rows / {pilot.document_id.nunique()} documents')

In [ ]:
display(Markdown('''## 1. 라벨 판단 요소 시그니처

각 요구사항에 비용·범위·책임·검수·완화 조건 표현이 있는지 탐색합니다. 이는 `통상수용/견적반영/계약·질의검토`를 자동 결정하는 규칙이 아니라, 어떤 판단 요소가 어느 문서와 유형에 집중되는지 확인하고 사람 감사 표본을 설계하기 위한 신호입니다.'''))

factor_cols = list(FACTOR_PATTERNS)
factor_summary = pd.DataFrame({
    '행수': df[factor_cols].sum(),
    '전체비율_pct': (df[factor_cols].mean() * 100).round(1),
}).sort_values('전체비율_pct', ascending=False)
display(factor_summary)

doc_factor = df.groupby('document_id')[factor_cols].mean() * 100
type_factor = df.groupby('eda_type_group', observed=True)[factor_cols].mean() * 100
cooccurrence = df[factor_cols].astype(int).T.dot(df[factor_cols].astype(int))
cooccurrence_rate = cooccurrence.div(df[factor_cols].sum().replace(0, np.nan), axis=0) * 100

fig, axes = plt.subplots(1, 3, figsize=(22, 7), gridspec_kw={'width_ratios': [1.2, 1.2, 1]})
sns.heatmap(doc_factor, annot=True, fmt='.0f', cmap='YlOrRd', ax=axes[0], cbar_kws={'label': '%'})
axes[0].set(title='문서별 판단 요소 포함률', xlabel='판단 요소', ylabel='document_id')
sns.heatmap(type_factor, annot=True, fmt='.0f', cmap='YlGnBu', ax=axes[1], cbar_kws={'label': '%'})
axes[1].set(title='EDA용 통합 유형별 판단 요소 포함률', xlabel='판단 요소', ylabel='EDA용 통합 유형')
sns.heatmap(cooccurrence_rate, annot=True, fmt='.0f', cmap='Purples', ax=axes[2], cbar_kws={'label': '조건부 동시출현률 (%)'})
axes[2].set(title='판단 요소 동시 출현', xlabel='함께 나타난 요소', ylabel='기준 요소')
plt.tight_layout()

multi_factor = df.loc[df['판단요소수'] >= 3, ['requirement_uid', 'document_id', 'eda_type_group', '판단요소수', 'requirement_name']].sort_values('판단요소수', ascending=False)
display(Markdown(f'**복합 판단 후보:** 판단 요소가 3개 이상인 요구사항은 {len(multi_factor):,}건입니다. 아래 표는 사람 감사와 프롬프트 경계 사례의 우선 후보입니다.'))
display(multi_factor.head(25))

In [ ]:
display(Markdown('''## 2. 요구사항 한 행만으로 판단 가능한가?

현재 라벨러와 후속 모델은 요구사항 한 행만 봅니다. 다른 절·별첨·요구사항 ID를 참조하거나, 핵심 범위를 협의로 유보하고, 불특정 표현이 겹치는 행은 국소 입력만으로 판단하기 어려울 수 있습니다. `문맥검토후보`는 정답 판정이 아니라 원문 문맥을 추가 확인할 감사 후보입니다.'''))

context_cols = list(CONTEXT_PATTERNS)
context_summary = pd.DataFrame({
    '행수': df[context_cols].sum(),
    '전체비율_pct': (df[context_cols].mean() * 100).round(1),
})
context_summary.loc['문맥검토후보'] = [df['문맥검토후보'].sum(), round(df['문맥검토후보'].mean() * 100, 1)]
display(context_summary.sort_values('전체비율_pct', ascending=False))

context_by_doc = (df.groupby('document_id')['문맥검토후보'].mean() * 100).sort_values()
context_by_type = (df.groupby('eda_type_group', observed=True)['문맥검토후보'].mean() * 100).sort_values()
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
context_by_doc.plot.barh(ax=axes[0], color='#E45756', title='문서별 문맥 검토 후보 비율')
context_by_type.plot.barh(ax=axes[1], color='#72B7B2', title='EDA용 통합 유형별 문맥 검토 후보 비율')
for ax in axes:
    ax.set_xlabel('비율 (%)')
plt.tight_layout()

context_examples = df.loc[
    df['문맥검토후보'],
    ['requirement_uid', 'document_id', 'eda_type_group', *context_cols, 'requirement_name', 'analysis_text'],
].copy()
context_examples['본문미리보기'] = context_examples['analysis_text'].str.slice(0, 220)
display(context_examples.drop(columns='analysis_text').sort_values(context_cols, ascending=False).head(30))
display(Markdown('**후속 검증:** 이 표에서 표본을 뽑아 원문 전체를 확인하고, 실제로 한 행만으로 판정 가능한지 사람이 이진 검토해야 합니다. 규칙 기반 비율을 그대로 데이터 결함률로 해석하면 안 됩니다.'))

In [ ]:
display(Markdown('''## 3. 비슷한 문구, 다른 판단 조건

문서 간 문자 n-gram TF-IDF 유사도를 계산하고, 가장 가까운 요구사항 쌍에서 판단 요소 시그니처가 다른 경우를 찾습니다. 이런 쌍은 단순 유사도 앵커가 잘못된 라벨을 끌어올 수 있는 경계 사례이자, 문서 단위 분할에도 남을 수 있는 표현 누수 후보입니다.'''))

vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, max_features=30_000, sublinear_tf=True)
matrix = vectorizer.fit_transform(df['analysis_text'])
similarity = cosine_similarity(matrix)
same_document = df['document_id'].to_numpy()[:, None] == df['document_id'].to_numpy()[None, :]
similarity[same_document] = -1.0
nearest_index = similarity.argmax(axis=1)
nearest_score = similarity[np.arange(len(df)), nearest_index]

pairs = []
seen = set()
signature_cols = factor_cols + context_cols
for left, right, score in zip(range(len(df)), nearest_index, nearest_score):
    right = int(right)
    key = tuple(sorted((left, right)))
    if key in seen:
        continue
    seen.add(key)
    differences = [name for name in signature_cols if bool(df.iloc[left][name]) != bool(df.iloc[right][name])]
    pairs.append({
        'similarity': round(float(score), 3),
        '조건차이수': len(differences),
        '다른조건': ', '.join(differences) or '없음',
        'left_uid': df.iloc[left]['requirement_uid'],
        'right_uid': df.iloc[right]['requirement_uid'],
        'left_name': df.iloc[left]['requirement_name'],
        'right_name': df.iloc[right]['requirement_name'],
    })
pair_df = pd.DataFrame(pairs)
conditional_pairs = pair_df.query('similarity >= 0.60 and 조건차이수 > 0').sort_values(['similarity', '조건차이수'], ascending=False)
boilerplate_pairs = pair_df.query('similarity >= 0.80 and 조건차이수 == 0').sort_values('similarity', ascending=False)

print(f'유사도≥0.60이며 조건 시그니처가 다른 쌍: {len(conditional_pairs):,}')
print(f'유사도≥0.80이며 조건 시그니처도 같은 반복 문구 후보: {len(boilerplate_pairs):,}')
display(Markdown('### 앵커 오선택 위험 후보'))
display(conditional_pairs.head(30))
display(Markdown('### 반복 표준 조항·평가 누수 후보'))
display(boilerplate_pairs.head(20))
display(Markdown('**읽는 법:** 조건 차이는 정규식 검출 차이일 뿐 실제 의미 차이를 확정하지 않습니다. 상위 쌍의 원문을 사람이 대조해 제공 주체·수량·비용·책임·검수 기준을 구조화하면 앵커 감사표로 발전시킬 수 있습니다.'))

In [ ]:
display(Markdown('''## 4. 40건 파일럿의 판단 공간 커버리지

파일럿은 전체 문서 행 비율을 그대로 복제할 필요가 없습니다. 대신 문서·도메인·EDA용 통합 유형·길이 구간·판단 요소·문맥 의존 후보·복합 시그니처를 얼마나 덮는지 확인해야 합니다. 대표 품질 추정용 표본과 경계 사례 중심 도전 표본은 별도로 보고하는 것이 원칙입니다.'''))

def category_coverage(population, sample, column):
    pop_counts = population[column].value_counts(dropna=False)
    sample_counts = sample[column].value_counts(dropna=False)
    covered = pop_counts.index.isin(sample_counts.index)
    return {
        '차원': column,
        '전체범주수': len(pop_counts),
        '파일럿포함범주수': int(covered.sum()),
        '범주커버리지_pct': round(covered.mean() * 100, 1),
        '누락범주': ', '.join(map(str, pop_counts.index[~covered])),
    }

coverage_dims = ['document_id', 'domain', 'eda_type_group', 'length_group']
category_report = pd.DataFrame([category_coverage(df, pilot, column) for column in coverage_dims])
display(category_report)

signal_cols = factor_cols + context_cols + ['문맥검토후보']
signal_coverage = pd.DataFrame({
    '전체행수': df[signal_cols].sum(),
    '파일럿행수': pilot[signal_cols].sum(),
    '전체비율_pct': (df[signal_cols].mean() * 100).round(1),
    '파일럿비율_pct': (pilot[signal_cols].mean() * 100).round(1),
})
signal_coverage['비율차이_pp'] = (signal_coverage['파일럿비율_pct'] - signal_coverage['전체비율_pct']).round(1)
display(signal_coverage.sort_values('비율차이_pp', key=lambda values: values.abs(), ascending=False))

def make_signature(frame):
    return frame[factor_cols].apply(lambda row: '|'.join(name for name in factor_cols if row[name]) or '신호없음', axis=1)

df['factor_signature'] = make_signature(df)
pilot['factor_signature'] = make_signature(pilot)
signature_report = df['factor_signature'].value_counts().rename('전체행수').to_frame()
signature_report['전체비율_pct'] = (signature_report['전체행수'] / len(df) * 100).round(1)
signature_report['파일럿행수'] = pilot['factor_signature'].value_counts().reindex(signature_report.index, fill_value=0)
signature_report['파일럿포함'] = signature_report['파일럿행수'] > 0
display(Markdown('### 전체에서 빈도가 높은 판단 요소 조합'))
display(signature_report.head(20))

uncovered_signatures = signature_report.query('파일럿포함 == False and 전체행수 >= 3')
addition_candidates = (
    df[df['factor_signature'].isin(uncovered_signatures.index)]
      .sort_values(['factor_signature', '판단요소수', 'char_len'], ascending=[True, False, False])
      .groupby('factor_signature', as_index=False).head(1)
      [['factor_signature', 'requirement_uid', 'document_id', 'eda_type_group', 'length_group', 'requirement_name']]
)
display(Markdown(f'''### 파일럿 보강 후보

전체에 3건 이상 존재하지만 파일럿에 없는 판단 요소 조합은 **{len(uncovered_signatures):,}개**입니다. 아래는 조합별 대표 후보 1건입니다.'''))
display(addition_candidates.head(30))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
signal_coverage[['전체비율_pct', '파일럿비율_pct']].plot.barh(ax=axes[0], title='판단·문맥 신호: 전체 vs 파일럿')
axes[0].set_xlabel('비율 (%)')
length_compare = pd.concat([
    (df['length_group'].value_counts(normalize=True) * 100).rename('전체'),
    (pilot['length_group'].value_counts(normalize=True) * 100).rename('파일럿'),
], axis=1).fillna(0)
length_compare.plot.bar(ax=axes[1], title='입력 길이 구간 커버리지')
axes[1].set_ylabel('비율 (%)')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()

display(Markdown('''### 다음 단계

1. 누락된 고빈도 판단 요소 조합과 문맥 검토 후보를 파일럿에 보강합니다.
2. 현재 40건을 **도전 표본**으로 사용할지, 전체 품질 추정을 위한 별도 층화 무작위 표본을 만들지 구분합니다.
3. 규칙 기반 후보 일부를 원문과 대조해 오탐률을 기록한 뒤 감사 표본 설계에만 사용합니다.
4. 라벨 생성 후에는 문서·유형·길이·판단 요소별 사람 일치율과 `계약·질의검토` Recall을 추가합니다.'''))